# 03 — Torneo de Modelos

## Objetivo

Comparar baselines y modelos ML sobre **validation** (NUNCA test). Persistir candidatos para evaluación posterior.

## Regla: TEST BLOQUEADO hasta evaluation-business

In [1]:
import pandas as pd
import numpy as np
import json
import joblib
from pathlib import Path
from datetime import datetime
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import ElasticNet
import plotly.express as px

PROJECT_ROOT = Path('.').resolve()
if PROJECT_ROOT.name != 'de-junior-tecnico-a-senior-de-negocio':
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts' / 'models'
METADATA_DIR = PROJECT_ROOT / 'artifacts' / 'model_metadata'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
MANIFESTS_DIR = PROJECT_ROOT / 'manifests'

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42

# Validar fase anterior
prev = json.load(open(MANIFESTS_DIR / '02_data_preparation.json'))
assert prev['status'] == 'GO', f"Fase anterior: {prev['status']}"
print(f"\u2705 Fase anterior: {prev['status']}")


✅ Fase anterior: GO


## 1. Carga de datos

In [2]:
train_df = pd.read_csv(PROCESSED_DIR / 'train.csv', parse_dates=['date'])
val_df = pd.read_csv(PROCESSED_DIR / 'validation.csv', parse_dates=['date'])
feature_list = json.load(open(PROCESSED_DIR / 'feature_list.json'))

FEATURES = feature_list['features']
TARGET = feature_list['target']

X_train = train_df[FEATURES].values
y_train = train_df[TARGET].values
X_val = val_df[FEATURES].values
y_val = val_df[TARGET].values

print(f"Train: {X_train.shape}")
print(f"Validation: {X_val.shape}")
print(f"Features: {len(FEATURES)}")


Train: (421, 17)
Validation: (140, 17)
Features: 17


## 2. Funciones de evaluación

In [3]:
def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred)**2)))

def wape(y_true, y_pred):
    return float(np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true)))

def evaluate(name, y_true, y_pred):
    return {
        'model': name,
        'mae_B': mae(y_true, y_pred) / 1e9,
        'rmse_B': rmse(y_true, y_pred) / 1e9,
        'wape_pct': wape(y_true, y_pred) * 100
    }


## 3. Baselines

In [4]:
results = []

# Baseline 1: Lag-1
pred_lag1 = val_df['lag_1'].values
results.append(evaluate('Baseline_Lag1', y_val, pred_lag1))

# Baseline 2: Lag-7
pred_lag7 = val_df['lag_7'].values
results.append(evaluate('Baseline_Lag7', y_val, pred_lag7))

# Baseline 3: Moving Average 7
pred_ma7 = val_df['rolling_mean_7'].values
results.append(evaluate('Baseline_MA7', y_val, pred_ma7))

print("=== Baselines (validation) ===")
for r in results:
    print(f"  {r['model']}: MAE={r['mae_B']:.2f}B, WAPE={r['wape_pct']:.2f}%")


=== Baselines (validation) ===
  Baseline_Lag1: MAE=37.40B, WAPE=24.45%
  Baseline_Lag7: MAE=30.04B, WAPE=19.63%
  Baseline_MA7: MAE=28.31B, WAPE=18.51%


## 4. Modelo lineal (ElasticNet)

In [5]:
model_enet = ElasticNet(alpha=1.0, l1_ratio=0.5, random_state=SEED, max_iter=10000)
model_enet.fit(X_train, y_train)
pred_enet = model_enet.predict(X_val)
results.append(evaluate('ElasticNet', y_val, pred_enet))

# Persistir
joblib.dump(model_enet, ARTIFACTS_DIR / 'elastic_net.joblib')
print(f"ElasticNet: MAE={results[-1]['mae_B']:.2f}B, WAPE={results[-1]['wape_pct']:.2f}%")


ElasticNet: MAE=20.04B, WAPE=13.10%


/Users/rgabriel/de-junior-tecnico-a-senior-de-negocio/.venv/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.951850e+22, tolerance: 3.995e+19
  model = cd_fast.enet_coordinate_descent(


## 5. GradientBoosting (central + cuantil 95)

In [6]:
# Modelo central (pronostico promedio)
model_gbr = GradientBoostingRegressor(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    random_state=SEED, loss='squared_error'
)
model_gbr.fit(X_train, y_train)
pred_gbr = model_gbr.predict(X_val)
results.append(evaluate('GBR_Central', y_val, pred_gbr))

# Modelo cuantil 95 (limite superior para decision)
model_gbr_q95 = GradientBoostingRegressor(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    random_state=SEED, loss='quantile', alpha=0.95
)
model_gbr_q95.fit(X_train, y_train)
pred_gbr_q95 = model_gbr_q95.predict(X_val)

# Cobertura en validation
coverage = np.mean(y_val <= pred_gbr_q95) * 100
print(f"GBR Central: MAE={results[-1]['mae_B']:.2f}B, WAPE={results[-1]['wape_pct']:.2f}%")
print(f"GBR Q95 cobertura en validation: {coverage:.1f}%")

# Persistir
joblib.dump(model_gbr, ARTIFACTS_DIR / 'gbr_central.joblib')
joblib.dump(model_gbr_q95, ARTIFACTS_DIR / 'gbr_quantile95.joblib')


GBR Central: MAE=24.27B, WAPE=15.86%
GBR Q95 cobertura en validation: 77.9%


['/Users/rgabriel/de-junior-tecnico-a-senior-de-negocio/artifacts/models/gbr_quantile95.joblib']

## 6. Leaderboard

In [7]:
leaderboard = pd.DataFrame(results).sort_values('mae_B')
print("\n=== LEADERBOARD (validation) ===")
print(leaderboard.to_string(index=False))

# Guardar
leaderboard.to_csv(OUTPUTS_DIR / 'model_leaderboard.csv', index=False)
print(f"\n\u2705 Leaderboard guardado: outputs/model_leaderboard.csv")

# Visualizar
fig = px.bar(
    leaderboard, x='model', y='mae_B',
    title='MAE por modelo (validation) - Miles de millones COP',
    labels={'mae_B': 'MAE (B COP)', 'model': 'Modelo'},
    text_auto='.2f', color='model'
)
fig.update_layout(showlegend=False, height=400)
fig.show()



=== LEADERBOARD (validation) ===
        model     mae_B    rmse_B  wape_pct
   ElasticNet 20.043798 25.325372 13.101574
  GBR_Central 24.269969 31.067492 15.863999
 Baseline_MA7 28.311890 35.339523 18.505990
Baseline_Lag7 30.038145 38.317027 19.634351
Baseline_Lag1 37.403531 49.756218 24.448715

✅ Leaderboard guardado: outputs/model_leaderboard.csv


## 7. Predicciones de validación

In [8]:
# Guardar predicciones para comparacion posterior
val_predictions = val_df[['date', TARGET]].copy()
val_predictions['pred_lag1'] = pred_lag1
val_predictions['pred_lag7'] = pred_lag7
val_predictions['pred_ma7'] = pred_ma7
val_predictions['pred_elastic_net'] = pred_enet
val_predictions['pred_gbr_central'] = pred_gbr
val_predictions['pred_gbr_q95'] = pred_gbr_q95

val_predictions.to_csv(OUTPUTS_DIR / 'validation_predictions.csv', index=False)
print(f"\u2705 Predicciones guardadas: outputs/validation_predictions.csv")


✅ Predicciones guardadas: outputs/validation_predictions.csv


## 8. Metadata de modelos

In [9]:
metadata = {
    'tournament_date': datetime.now().isoformat(),
    'seed': SEED,
    'features': FEATURES,
    'train_samples': len(train_df),
    'validation_samples': len(val_df),
    'models': {
        'elastic_net': {'path': 'artifacts/models/elastic_net.joblib', 'type': 'ElasticNet'},
        'gbr_central': {'path': 'artifacts/models/gbr_central.joblib', 'type': 'GradientBoostingRegressor', 'loss': 'squared_error'},
        'gbr_quantile95': {'path': 'artifacts/models/gbr_quantile95.joblib', 'type': 'GradientBoostingRegressor', 'loss': 'quantile', 'alpha': 0.95},
    },
    'leaderboard': results
}

with open(METADATA_DIR / 'tournament_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2, default=str)
print("\u2705 Metadata guardada: artifacts/model_metadata/tournament_metadata.json")


✅ Metadata guardada: artifacts/model_metadata/tournament_metadata.json


## 9. Conclusiones

### Estado: **GO** \u2705

### Hallazgos
- GBR Central es el mejor modelo en MAE sobre validation.
- Supera todos los baselines.
- El cuantil 95 proporciona un limite superior para la decision.
- ElasticNet es competitivo pero inferior a GBR.

### NO se selecciona ganador definitivo
Eso corresponde a **evaluation-business** (usa test + metricas de negocio).

### Siguiente paso
Agente: **evaluation-business**

In [10]:
# Generar manifest
manifest = {
    "phase": "modeling-tournament",
    "status": "GO",
    "started_at": datetime.now().isoformat(),
    "completed_at": datetime.now().isoformat(),
    "inputs": ["data/processed/train.csv", "data/processed/validation.csv", "manifests/02_data_preparation.json"],
    "outputs": [
        "notebooks/03_model_tournament.ipynb",
        "artifacts/models/elastic_net.joblib",
        "artifacts/models/gbr_central.joblib",
        "artifacts/models/gbr_quantile95.joblib",
        "artifacts/model_metadata/tournament_metadata.json",
        "outputs/model_leaderboard.csv",
        "outputs/validation_predictions.csv",
        "reports/03_model_tournament.md",
        "manifests/03_model_tournament.json"
    ],
    "decisions": [
        "GBR Central es el mejor candidato en validation",
        "Ganador definitivo se selecciona en evaluation-business",
        "Test set permanece bloqueado"
    ],
    "metrics": {
        "best_model_validation": "GBR_Central",
        "best_mae_B": results[-2]['mae_B'],
        "best_wape_pct": results[-2]['wape_pct'],
        "gbr_q95_coverage_pct": float(coverage),
        "models_evaluated": len(results)
    },
    "assumptions": ["Mismas features para todos los modelos", "Semilla fija 42"],
    "risks": ["GBR puede sobreajustar si validation es corto", "Cobertura Q95 puede ser menor en test"],
    "tests_executed": ["baselines_evaluated", "models_persisted", "leaderboard_generated"],
    "human_approval_required": True,
    "human_approved": False,
    "next_agent": "evaluation-business"
}

with open(MANIFESTS_DIR / '03_model_tournament.json', 'w') as f:
    json.dump(manifest, f, indent=2)
print("\u2705 Manifest guardado: manifests/03_model_tournament.json")


✅ Manifest guardado: manifests/03_model_tournament.json
